# Delete the Chameleon environment

This notebook deletes the selected VM, its floating IP, any disposable boot volume, and its lease. It retains the shared security groups and cached S3 credentials for later launches.

## Select the environment type

In [ ]:
DEPLOYMENT_TYPE = "cpu"  # "cpu" or "gpu"

## Select the Chameleon project

In [ ]:
from chi import context, lease, server
import chi
import os
import time

if DEPLOYMENT_TYPE not in {"cpu", "gpu"}:
    raise ValueError("DEPLOYMENT_TYPE must be 'cpu' or 'gpu'")

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
RESOURCE_PREFIX = f"jupyter-opencode-{DEPLOYMENT_TYPE}-{username}"
LEASE_NAME = f"lease-{RESOURCE_PREFIX}"
SERVER_NAME = f"node-{RESOURCE_PREFIX}"
BOOT_VOLUME_NAME = f"boot-vol-{RESOURCE_PREFIX}"

print(f"Server: {SERVER_NAME}")
print(f"Lease: {LEASE_NAME}")

## Delete the server and floating IP

In [ ]:
existing_servers = [item for item in server.list_servers() if item.name == SERVER_NAME]
if existing_servers:
    s = existing_servers[0]
    s.delete(idempotent=True, delete_ips=True)
    print(f"Deleting server {SERVER_NAME} and its floating IP")
else:
    print(f"Server {SERVER_NAME} does not exist")

os_conn = chi.clients.connection()
deadline = time.time() + 600
while os_conn.compute.find_server(SERVER_NAME, ignore_missing=True):
    if time.time() >= deadline:
        raise TimeoutError(f"Timed out waiting for {SERVER_NAME} to be deleted")
    time.sleep(10)
print("Server deleted")

## Confirm the disposable boot volume is deleted

A boot volume created by a launch notebook has `delete_on_termination=True`. This cell removes it if OpenStack left it behind after deleting the VM.

In [ ]:
cinder_client = chi.clients.cinder()
deadline = time.time() + 600
while True:
    remaining_volumes = [v for v in cinder_client.volumes.list() if v.name == BOOT_VOLUME_NAME]
    if not remaining_volumes:
        break
    for volume in remaining_volumes:
        volume = cinder_client.volumes.get(volume.id)
        if volume.status == "available":
            cinder_client.volumes.delete(volume.id)
            print(f"Deleting leftover boot volume {volume.name}")
        elif volume.status in {"error", "error_deleting"}:
            raise RuntimeError(f"Boot volume {volume.name} is in {volume.status}")
    if time.time() >= deadline:
        raise TimeoutError(f"Timed out waiting for {BOOT_VOLUME_NAME} to be deleted")
    time.sleep(10)
print("Disposable boot volume deleted or not present")

## Delete the lease

In [ ]:
matching_leases = [item for item in lease.list_leases() if item.name == LEASE_NAME]
if matching_leases:
    l = matching_leases[0]
    l.delete()
    print(f"Deleted lease {LEASE_NAME}")
else:
    print(f"Lease {LEASE_NAME} does not exist")

print("Cleanup complete. S3 credentials and shared security groups were retained.")